In [2]:
from collections import OrderedDict
import torch

input_size = 5
hidden_dims = 10
output_size = 2

net = torch.nn.Sequential(
    OrderedDict(
        [
            ("layer1", torch.nn.Linear(input_size, hidden_dims)),
            ("layer2", torch.nn.Linear(hidden_dims, output_size)),
        ]
    )
).requires_grad_(False)

In [3]:
from nnsight import NNsight

tiny_model = NNsight(net)

/opt/miniconda3/envs/new-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print(tiny_model)

Sequential(
  (layer1): Linear(in_features=5, out_features=10, bias=True)
  (layer2): Linear(in_features=10, out_features=2, bias=True)
)


In [5]:
# random input
input = torch.rand((1, input_size))

with tiny_model.trace(input) as tracer:
    pass

with tiny_model.trace(input) as tracer:

    output = tiny_model.output.save()

print(output)

tensor([[ 0.0258, -0.1953]])


In [6]:

with tiny_model.trace(input) as tracer:

    l1_output = tiny_model.layer1.output.save()
    l2_input = tiny_model.layer2.input.save()

print(l1_output)
print(l2_input)

tensor([[-0.2419,  0.2552,  0.9497, -0.3817,  0.4257, -0.0041,  0.0494, -0.5149,
          0.0462,  0.0250]])
tensor([[-0.2419,  0.2552,  0.9497, -0.3817,  0.4257, -0.0041,  0.0494, -0.5149,
          0.0462,  0.0250]])


In [7]:
with tiny_model.trace(input) as tracer:
  print(tiny_model.input)
  print("Layer 1 - out: ", tiny_model.layer1.output)

tensor([[0.4648, 0.9030, 0.7958, 0.8277, 0.2718]])
Layer 1 - out:  tensor([[-0.2419,  0.2552,  0.9497, -0.3817,  0.4257, -0.0041,  0.0494, -0.5149,
          0.0462,  0.0250]])


In [8]:
with tiny_model.trace(input):

    # Note we don't need to call .save() on the output,
    # as we're only using its value within the tracing context.
    l1_output = tiny_model.layer1.output

    # We do need to save the argmax tensor however,
    # as we're using it outside the tracing context.
    l1_amax = torch.argmax(l1_output, dim=1).save()

    value = (tiny_model.layer1.output.sum() + tiny_model.layer2.output.sum()).save()

print(l1_amax[0])

print(value)

tensor(2)
tensor(0.4391)


In [9]:

# Take a tensor and return the sum of its elements
def tensor_sum(tensor):
    flat = tensor.flatten()
    total = 0
    for element in flat:
        total += element.item()

    return torch.tensor(total)

with tiny_model.trace(input) as tracer:

    # call on our custom function within the trace context
    custom_sum = tensor_sum(tiny_model.layer1.output[0]).save()
    sum = tiny_model.layer1.output.sum().save()


print(custom_sum, sum)

tensor(0.6086) tensor(0.6086)


In [10]:
with tiny_model.trace(input):

    # Save the output before the edit to compare.
    # Notice we apply .clone() before saving as the setting operation is in-place.
    l1_output_before = tiny_model.layer1.output.clone().save()

    # Access the 0th index of the hidden state dimension and set it to 0.
    tiny_model.layer1.output[:, 0] = 0

    # Save the output after to see our edit.
    l1_output_after = tiny_model.layer1.output.save()

print("Before:", l1_output_before)
print("After:", l1_output_after)

Before: tensor([[-0.2419,  0.2552,  0.9497, -0.3817,  0.4257, -0.0041,  0.0494, -0.5149,
          0.0462,  0.0250]])
After: tensor([[ 0.0000,  0.2552,  0.9497, -0.3817,  0.4257, -0.0041,  0.0494, -0.5149,
          0.0462,  0.0250]])


In [11]:

# Now in NNsight 0.5
with tiny_model.trace(input):
  # 1) access l1 & l2 outputs so trace knows these are intermediate values we care about
  l1_output = tiny_model.layer1.output
  # 2) make sure gradient flows back to l1 (it will pass by l2)
  l1_output.requires_grad = True
  l2_output = tiny_model.layer2.output

  # 3) access gradients within a backwards trace
  with tiny_model.output.sum().backward():
    # access .grad within backward context in REVERSE ORDER
    layer2_output_grad = l2_output.grad.save()
    layer1_output_grad = l1_output.grad.save()

print("Layer 1 output gradient:", layer1_output_grad)
print("Layer 2 output gradient:", layer2_output_grad)

Layer 1 output gradient: tensor([[-0.1552, -0.1855, -0.3570, -0.2112, -0.0025,  0.0716,  0.0516, -0.0092,
         -0.2552,  0.1742]])
Layer 2 output gradient: tensor([[1., 1.]])


In [12]:
# Now in NNsight 0.5
with tiny_model.trace(input):
  # 1) access l1 & l2 outputs so trace knows these are intermediate values we care about
  l1_output = tiny_model.layer1.output
  # 2) make sure gradient flows back to l1 (it will pass by l2)
  l1_output.requires_grad = True
  l2_output = tiny_model.layer2.output

  # 3) access gradients within a backwards trace
  with tiny_model.output.sum().backward():
    # access .grad within backward context in REVERSE ORDER
    l2_output.grad = l2_output.grad * 2
    layer2_output_grad = l2_output.grad.save()
    layer1_output_grad = l1_output.grad.save()

print("Layer 1 output gradient:", layer1_output_grad)
print("Layer 2 output gradient:", layer2_output_grad)

Layer 1 output gradient: tensor([[-0.3103, -0.3711, -0.7140, -0.4225, -0.0050,  0.1433,  0.1032, -0.0184,
         -0.5104,  0.3485]])
Layer 2 output gradient: tensor([[2., 2.]])


In [13]:

with tiny_model.trace(input) as tracer:
   l1_out = tiny_model.layer1.output.save()
   tracer.stop()

# get the output of the first layer and stop tracing
print("L1 - Output: ", l1_out)

L1 - Output:  tensor([[-0.2419,  0.2552,  0.9497, -0.3817,  0.4257, -0.0041,  0.0494, -0.5149,
          0.0462,  0.0250]])


In [14]:
from nnsight import LanguageModel

llm = LanguageModel("openai-community/gpt2", device_map="auto", dispatch=True)

print(llm)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
  (generator): Generator(
    (streamer): Streamer()
  )
)


In [15]:
with llm.trace("The Eiffel Tower is in the city of"):
    # Access the last layer using h[-1] as it's a ModuleList
    # Access the first index of .output as that's where the hidden states are.
    llm.transformer.h[-1].mlp.output[0][:] = 0

    # # Logits come out of model.lm_head and we apply argmax to get the predicted token ids.
    token_ids = llm.lm_head.output.argmax(dim=-1).save()

print("\nToken IDs:", token_ids)

# Apply the tokenizer to decode the ids into words after the tracing context.
print("Prediction:", llm.tokenizer.decode(token_ids[0][-1]))


Token IDs: tensor([[ 262,   12,  417, 8765,   11,  257,  262, 3504,  338, 3576]],
       device='mps:0')
Prediction:  London


In [16]:
with llm.trace() as tracer:

    with tracer.invoke("The Eiffel Tower is in the city of"):

        # Ablate the last MLP for only this batch.
        llm.transformer.h[-1].mlp.output[0][:] = 0

        # Get the output for only the intervened on batch.
        token_ids_intervention = llm.lm_head.output.argmax(dim=-1).save()

    with tracer.invoke("The Eiffel Tower is in the city of"):

        # Get the output for only the original batch.
        token_ids_original = llm.lm_head.output.argmax(dim=-1).save()


print("Original token IDs:", token_ids_original)
print("Modified token IDs:", token_ids_intervention)

print("Original prediction:", llm.tokenizer.decode(token_ids_original[0][-1]))
print("Modified prediction:", llm.tokenizer.decode(token_ids_intervention[0][-1]))

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Original token IDs: tensor([[ 198,   12,  417, 8765,  318,  257,  262, 3504, 7372, 6342]],
       device='mps:0')
Modified token IDs: tensor([[ 262,   12,  417, 8765,   11,  257,  262, 3504,  338, 3576]],
       device='mps:0')
Original prediction:  Paris
Modified prediction:  London


In [17]:
with llm.trace() as tracer:
    barrier = tracer.barrier(2)
    with tracer.invoke("The Eiffel Tower is in the city of"):
        embeddings = llm.transformer.wte.output
        # call barrier
        barrier()

    with tracer.invoke("_ _ _ _ _ _ _ _ _ _"):
        # tell model to wait for the output from the previous invoke with barrier
        barrier()
        llm.transformer.wte.output = embeddings
        token_ids_intervention = llm.lm_head.output.argmax(dim=-1).save()

    with tracer.invoke("_ _ _ _ _ _ _ _ _ _"):
        token_ids_original = llm.lm_head.output.argmax(dim=-1).save()

print("original prediction shape", token_ids_original[0][-1].shape)
print("Original prediction:", llm.tokenizer.decode(token_ids_original[0][-1]))

print("modified prediction shape", token_ids_intervention[0][-1].shape)
print("Modified prediction:", llm.tokenizer.decode(token_ids_intervention[0][-1]))

original prediction shape torch.Size([])
Original prediction:  _
modified prediction shape torch.Size([])
Modified prediction:  Paris


In [18]:
# using .all():
prompt = 'The Eiffel Tower is in the city of'
layers = llm.transformer.h
n_new_tokens = 50
with llm.generate(prompt, max_new_tokens=n_new_tokens) as tracer:
    hidden_states = list().save() # Initialize & .save() list

    # Call .all() to apply intervention to each new token
    with tracer.all():

        # Apply intervention - set first layer output to zero
        layers[0].output[0][:] = 0

        # Append desired hidden state post-intervention
        hidden_states.append(layers[-1].output) # no need to call .save

print("Hidden state length: ",len(hidden_states))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


Hidden state length:  50


In [19]:
# using .all():
prompt = 'The Eiffel Tower is in the city of'
layers = llm.transformer.h
n_new_tokens = 50
with llm.generate(prompt, max_new_tokens=n_new_tokens) as tracer:
    hidden_states = list().save() # Initialize & .save() list

    # Call .all() to apply intervention to each new token
    with tracer.iter[2:5]:

        # Apply intervention - set first layer output to zero
        layers[0].output[0][:] = 0

        # Append desired hidden state post-intervention
        hidden_states.append(layers[-1].output) # no need to call .save

print("Hidden state length: ",len(hidden_states))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Hidden state length:  0


In [20]:
# we take the hidden states with the expected output "Paris"
with llm.trace("The Eiffel Tower is located in the city of") as tracer:
    hs11 = llm.transformer.h[11].output[0][:, -1, :].save()

# the edited model will now always predict "Paris" as the next token
with llm.edit() as llm_edited:
    llm.transformer.h[11].output[0][:, -1, :] = hs11

# we demonstrate this by comparing the output of an unmodified model...
with llm.trace("Vatican is located in the city of") as tracer:
    original_tokens = llm.lm_head.output.argmax(dim=-1).save()

# ...with the output of the edited model
with llm_edited.trace("Vatican is located in the city of") as tracer:
    modified_tokens = llm.lm_head.output.argmax(dim=-1).save()


print("\nOriginal Prediction: ", llm.tokenizer.decode(original_tokens[0][-1]))
print("Modified Prediction: ", llm.tokenizer.decode(modified_tokens[0][-1]))


Original Prediction:   Rome
Modified Prediction:   Paris


In [21]:
# we use the hidden state we saved above (hs11)
with llm.edit(inplace=True) as llm_edited:
    llm.transformer.h[11].output[0][:, -1, :] = hs11

# we demonstrate this by comparing the output of an unmodified model...
with llm.trace("Vatican is located in the city of") as tracer:
    modified_tokens = llm.lm_head.output.argmax(dim=-1).save()

print("Modified In-place: ", llm.tokenizer.decode(modified_tokens[0][-1]))

Modified In-place:   Paris


In [ ]:
llm.clear_edits()

with llm.trace("Vatican is located in the city of"):
    modified_tokens = llm.lm_head.output.argmax(dim=-1).save()

print("Edits cleared: ", llm.tokenizer.decode(modified_tokens[0][-1]))

Edits cleared:   Rome
